# ▶ Data builder for arch 3 (and 4): space-attached BPE + 393M training tokens
Runs on a **CPU session** (Settings → Accelerator **None**): no GPU quota used. Internet **On**, Secret `HF_TOKEN`.

1. **Save Version → Save & Run All (Commit)** and close the browser.
2. When done, from the version's Output create a **Kaggle Dataset** named `data-sbpe8000` from the folder `/kaggle/working/data_sbpe8000` (8 files: tokenizer, train.bin, val.bin, meta.json, dev prompts/refs).
3. Attach that dataset to the arch-3 notebook (its `prebuilt_dir` expects `/kaggle/input/datasets/parishrishah/data-sbpe8000`).

What it does: streams the 4 sources (equal mix), holds out 2% of documents per source, trains the **space-attached BPE** (vocab 8000, 30M chars, fast trainer) and encodes ~393M training tokens (3x arch 1: enough for 12,000 steps without repeating data).


# Phase 1 — Legal Language Model (from scratch) · v2
**CS781 Capstone · Kaggle GPU notebook · Arch #1 decoder-only (`arch="decoder"`) · Arch #2 encoder-decoder (`arch="encdec"`) · Arch #3 prefix-LM (`arch="prefixlm"`)**

End-to-end pipeline for one Phase-1 experiment:

1. Stream the gated `Exploration-Lab/CS-781-Capstone` Phase-1 data (no 19 GB download).
2. Split **by document** into train / held-out validation, and build a **local dev set** (512-word prompt → 128-word reference) from the held-out documents.
3. Train **your A1 byte-level BPE** tokenizer, encode the corpus in parallel, save `train.bin` / `val.bin`, and reuse them in every architecture notebook (same data, so the comparison is controlled).
4. Build **your A3 DecoderLM** (50M–300M params), train it from scratch with resumable checkpoints.
5. Evaluate locally (val loss/perplexity + ROUGE-L / BLEU-4 / BERTScore on the dev set) and try different decoding settings.
6. Generate exactly 128 words for each official test prompt, check the output, and write `prediction.json` → `submission.zip`.
7. Log the run to `experiments.jsonl` and export the checkpoint + code for Phases 2–7.

---
### What changed from draft-1 (and why) — you must be able to explain each of these in the viva
| # | Problem seen in draft-1 output | Fix |
|---|---|---|
| FIX-1 | Smoke-test val loss was **15.2 → 10.4**, which is *higher* than `ln(2000)=7.6` (the loss of a uniform guess). Cause: no weight init. `nn.Embedding` defaults to N(0,1) and is **tied** to the output head, so initial logits have std ≈ √d_model ≈ 16, which makes the loss very large. That is why the output was "the the the …". | GPT-style init N(0, 0.02) for Linear/Embedding, plus a check that the initial loss is ≈ ln(V). |
| FIX-2 | Once embeddings are N(0, 0.02), the token signal is ~50× smaller than the sinusoidal position signal (amplitude ~1). | Multiply token embeddings by √d_model, as in the original Transformer. One marked line in `model_arch.py`, controlled by `scale_emb`. |
| FIX-3 | A3 default is **post-LN** (`norm_first=False`). Post-LN often trains unstably at 10+ layers without very careful warmup (Xiong et al. 2020). | `norm_first=True` (pre-LN). This flag already exists in your A3 code. |
| FIX-4 | Word counts were **78–80**, but Phase 1 needs 128 words. Generation used a fixed token cap and also stopped at `</s>`. | Generate until ≥129 words, then cut to 128. Special tokens are masked during sampling. |
| FIX-5 | Generating one prompt at a time took 12 min even for the tiny model. With the real model it would take hours. | Batched generation. Prompts in a batch are cut to the same token length, so no padding mask is needed. With **GPU T4 x2**, batches are split across **both GPUs** (about 2x faster; identical output). Progress prints at least every 5 min. |
| FIX-6 | `hf_hub_download` downloaded all ~19 GB of training JSON even in smoke mode. | Stream with `HfFileSystem`, so only the bytes you use are read. |
| FIX-7 | The config used 20M train tokens but consumed 4000×32×1024 ≈ 131M tokens, so the same data was seen **~6.5 times**. Validation was the token tail of the stream (train and val could share a document). | Split by document, **stratified per source** (equal val tokens and dev prompts from each of the 4 sources; val loss also reported per source). The data budget (`train_tokens`) is a **fixed** number (= what arch-1 consumes), independent of each architecture's steps/batch, so every architecture gets the *identical* split and tokenizer. A tokenizer fingerprint is checked on resume. |
| FIX-8 | Validation used 40 *random* batches, so it was noisy and not comparable across architectures. | Fixed, evenly spaced validation windows (the same for every run). |
| FIX-9 | Only the best weights were saved. A Kaggle timeout or disconnect lost the whole run. | Resumable `ckpt_last.pt` (model + optimizer + scaler + step + RNG), plus a **session clock**: training stops early enough that generation + zip always finish inside Kaggle's 12 h limit. Test predictions are written *before* the optional dev evaluation. |
| FIX-10 | No local evaluation (the document, Sec 16, asks for it). | Local dev set built from held-out documents, scored with ROUGE-L, BLEU-4 and BERTScore. |
| misc | Weight decay was also applied to LayerNorm and bias. LR decayed all the way to 0. The prompt could end mid-word. | Separate no-decay parameter group, `min_lr_ratio`, and a trailing space after the prompt. |

### How to run on Kaggle
1. Settings → Accelerator **GPU T4 x2** (training uses one GPU, generation uses both), Internet **On**. Add the Secret `HF_TOKEN`.
2. `SMOKE_TEST=True` first (~15 min). Check: initial loss ≈ ln(V), word counts = 128, zip passes the checks.
3. `SMOKE_TEST=False` → **Save Version → Save & Run All (Commit)**. Commit mode runs in the background for up to 12 h.
4. When it finishes, create a **Kaggle Dataset from the output**. Point `prebuilt_dir` at its `data_*` folder in the other architecture notebooks, and `resume_from` at `ckpt_last.pt` if a run was cut off.

In [ ]:
# 1 · Install dependencies (torch is already on Kaggle). Requires Internet = ON.
!pip -q install -U "huggingface_hub>=0.23" ijson rouge-score sacrebleu bert-score

In [ ]:
# 2 · Configuration ------------------------------------------------------------
RUN_NAME   = "data_sbpe8000"             # names the run folder, checkpoints, logs and zip
SMOKE_TEST = False                       # True = tiny end-to-end check; False = real run

CONFIG = dict(
    hf_repo   = "Exploration-Lab/CS-781-Capstone",
    phase_dir = "Phase_1_Pretraining",
    sources   = ["acts", "high_courts", "supreme_court", "tribunals"],
    mix       = {"acts": 1, "high_courts": 1, "supreme_court": 1, "tribunals": 1},  # relative char-budget per source (a data-mixing experiment)
    prebuilt_dir = None,        # e.g. "/kaggle/input/<your-dataset>/data_bpe8000" -> reuse tokenizer + tokens (skip steps 5-6)

    # ---- tokenizer ----
    tokenizer_type  = "space",     # "a1" = your A1 BPE (every space is its own token, ~2.3 tokens/word)
                                # "space" = space-attached BPE (" the", ~1.3 tokens/word -> whole prompt fits)
    vocab_size      = 8000,
    tok_train_chars = 30_000_000,

    # ---- data budget / validation ----
    train_tokens   = 393_217_025,  # FIXED data budget (= 4000 steps x 32 x 1024 + 1025, what arch-1 uses).
                                   # Do NOT change between architectures: it defines the train/val split.
    val_tokens     = 2_000_000,
    val_doc_frac   = 0.02,      # fraction of documents per source held out (never trained on)
    n_dev_examples = 100,       # local dev prompts built from held-out docs

    # ---- model (your A3 DecoderLM) ----
    d_model=768, n_layers=10, n_heads=12, n_kv_heads=12, d_ff=3072,
    max_seq_len=1024, dropout=0.1, pos_emb_type="sinusoidal",
    norm_first = True,          # [FIX-3] pre-LN
    scale_emb  = True,          # [FIX-2] sqrt(d_model) token-embedding scale

    # ---- architecture switch ----
    arch = "decoder",           # "decoder" = arch-1 | "encdec" = arch-2 encoder-decoder | "prefixlm" = arch-3
    # arch-2 only (uses d_model, n_heads, d_ff, dropout above):
    enc_layers=6, dec_layers=4, # encoder / decoder depth
    enc_len=768, dec_len=384,   # window split: encoder reads 768 prompt tokens, decoder predicts 384
    # arch-3 only (uses d_model, n_layers, n_heads, n_kv_heads, max_seq_len, dropout above):
    ffn_hidden=2048,            # SwiGLU hidden size
    prefix_min=0.0, prefix_max=0.85,  # training prefix length p ~ U[prefix_min, prefix_max] * max_seq_len
    eval_prefix=0.75,           # fixed prefix fraction for the validation loss

    # ---- training ----
    batch_size=8, grad_accum=4, lr=3e-4, min_lr_ratio=0.1, weight_decay=0.1,
    warmup_steps=200, max_steps=4000, grad_clip=1.0,
    eval_every=250, eval_batches=40, ckpt_every=250,
    session_limit_hours = 11.5, # Kaggle kills a session at 12 h (counted from the FIRST cell, incl. data prep)
    reserve_hours_after_training = 3.0,  # time kept for test generation (~2-2.5 h) + zip + dev eval
    resume_from = None,         # path to a previous ckpt_last.pt (e.g. /kaggle/input/.../ckpt_last.pt)

    # ---- generation ----
    gen_words=128, max_new_tokens=1024, gen_batch_size=16, gen_ctx_tokens=None,  # cap only; stops early at 128 words
    temperature=0.8, top_k=50, top_p=0.95, repetition_penalty=1.0,             # 1.0 = off
    gen_gpus = None,            # None = use every visible GPU for generation (Kaggle 'GPU T4 x2' -> 2x faster)
    run_bertscore = True,       # local BERTScore on the dev set (downloads roberta-large once)
    gen_reserve_tokens = 256,   # prefix-LM: keep at most max_seq_len - this prompt tokens (room for the KV cache)
    decode_search = False,      # True: try decode_grid on dev prompts BEFORE the test set, use the best BERTScore
    decode_search_n = 60,       # dev prompts per decoding setting (balanced over the 4 sources)
    decode_grid = [
        dict(name="t0.8_p0.95",        temperature=0.8, top_k=50, top_p=0.95, repetition_penalty=1.0),
        dict(name="t0.7_p0.95",        temperature=0.7, top_k=50, top_p=0.95, repetition_penalty=1.0),
        dict(name="t0.6_p0.9",         temperature=0.6, top_k=50, top_p=0.90, repetition_penalty=1.0),
        dict(name="t0.7_p0.95_rep1.1", temperature=0.7, top_k=50, top_p=0.95, repetition_penalty=1.1),
        dict(name="t0.6_p0.9_rep1.15", temperature=0.6, top_k=50, top_p=0.90, repetition_penalty=1.15),
        dict(name="greedy_rep1.2",     temperature=0.0, top_k=0,  top_p=1.0,  repetition_penalty=1.2),
    ],

    seed=1337, out_dir="/kaggle/working",
)

if SMOKE_TEST:
    CONFIG.update(
        vocab_size=2000, tok_train_chars=1_000_000, train_tokens=246_017, val_tokens=50_000, val_doc_frac=0.05, n_dev_examples=16,
        d_model=256, n_layers=4, n_heads=4, n_kv_heads=4, d_ff=1024, max_seq_len=256,
        enc_layers=2, dec_layers=2, enc_len=128, dec_len=128, ffn_hidden=688,
        gen_reserve_tokens=96, decode_search_n=8,
        batch_size=8, grad_accum=1, warmup_steps=20, max_steps=120,
        eval_every=40, eval_batches=10, ckpt_every=40, run_bertscore=False,
    )

import os, random, math, json, time, pickle, zipfile, shutil
SESSION_START = time.time()                      # the 12 h Kaggle clock starts (roughly) here
def session_hours(): return (time.time() - SESSION_START) / 3600
from dataclasses import asdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"

OUT_ROOT = CONFIG["out_dir"]
RUN_DIR  = os.path.join(OUT_ROOT, "runs", RUN_NAME + ("_smoke" if SMOKE_TEST else ""))
DATA_DIR = CONFIG["prebuilt_dir"] or os.path.join(
    OUT_ROOT, f"data_{'sbpe' if CONFIG['tokenizer_type'] == 'space' else 'bpe'}{CONFIG['vocab_size']}"
    + ("_smoke" if SMOKE_TEST else ""))
os.makedirs(RUN_DIR, exist_ok=True)
SEQ_LEN = CONFIG["enc_len"] + CONFIG["dec_len"] if CONFIG["arch"] == "encdec" else CONFIG["max_seq_len"]
TOKENS_PER_STEP = CONFIG["batch_size"] * CONFIG["grad_accum"] * SEQ_LEN
TRAIN_TOKENS = CONFIG["train_tokens"]
print("Device:", device, "| SMOKE_TEST:", SMOKE_TEST, "| run dir:", RUN_DIR, "| data dir:", DATA_DIR)
print(f"Tokens per optimizer step: {TOKENS_PER_STEP:,} | tokens the run will consume: {CONFIG['max_steps']*TOKENS_PER_STEP:,}")

In [ ]:
# 3 · Hugging Face login (the dataset is gated) --------------------------------
# Accept the dataset terms once at https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone
from huggingface_hub import login
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    print("Loaded HF token from Kaggle Secrets.")
except Exception:
    pass
if not token:
    import getpass
    token = os.environ.get("HF_TOKEN") or getpass.getpass("Enter your Hugging Face token: ")
login(token=token)
print("Logged in to Hugging Face.")

## 4 · Your Assignment code (A3 model + A1 tokenizer)
`rope.py`, `attention.py` and `a1_tokenizer.py` are your code, unchanged. `model_arch.py` has one marked change (**[FIX-2]**: a `scale_emb` flag, off by default).

In [ ]:
import os
os.makedirs('a3src', exist_ok=True)

In [ ]:
%%writefile a3src/rope.py
# TODO 3.1: rotary positional embeddings from scratch (split-half convention).
import torch


def build_rope_cache(max_seq_len, d_head, base=10000.0):
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))   # (d_head/2,)
    positions = torch.arange(max_seq_len).float()
    angles = torch.outer(positions, inv_freq)                                  # (T, d_head/2)
    angles = torch.cat([angles, angles], dim=-1)                               # (T, d_head)
    return angles.cos(), angles.sin()


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin

In [ ]:
%%writefile a3src/attention.py
# TODO 3.2: MHA / GQA / MQA attention with optional RoPE applied to q and k.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0,
                 use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        if attention_scale is None:
            self.attention_scale = 1.0 / math.sqrt(self.d_head)
        else:
            self.attention_scale = float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            q = apply_rope(q, self.rope_cos[:T], self.rope_sin[:T])
            k = apply_rope(k, self.rope_cos[:T], self.rope_sin[:T])

        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.attention_scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(F.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, self.n_heads * self.d_head)
        return self.resid_dropout(self.out_proj(out))

In [ ]:
%%writefile a3src/model_arch.py
# TODO 3.2: pass use_rope through the decoder; only build sinusoidal PE when requested.
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from attention import CausalSelfAttention


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None
    norm_first: bool = False
    scale_emb: bool = False            # [FIX-2] multiply token embeddings by sqrt(d_model) (Vaswani et al. 2017, Sec 3.4)


def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm_first = cfg.norm_first
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.cfg.scale_emb:             # [FIX-2]
            x = x * math.sqrt(self.cfg.d_model)
        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T].unsqueeze(0)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)

In [ ]:
%%writefile a3src/encdec.py
# Architecture 2: Encoder-decoder Transformer for "prompt -> continuation" language modelling.
#
# Every training window of enc_len + dec_len contiguous tokens is split in two:
#   encoder  <- first enc_len tokens (the "prompt"), bidirectional self-attention
#   decoder  <- next dec_len tokens (the "continuation"), causal self-attention + cross-attention
# The loss is computed on the continuation tokens only, which is exactly the Phase-1 test task.
# Positions: RoPE (your A3 rope.py) inside self-attention; no positional signal in cross-attention.
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


@dataclass
class EncDecConfig:
    vocab_size: int
    d_model: int = 768
    n_heads: int = 12
    d_ff: int = 3072
    enc_layers: int = 6
    dec_layers: int = 4
    enc_len: int = 768           # prompt tokens the encoder reads
    dec_len: int = 384           # continuation tokens the decoder predicts per window
    dropout: float = 0.1
    bos_id: int = 2              # decoder start token (<s> in your A1 tokenizer)

    @property
    def max_seq_len(self):       # length of one training window (used by the data pipeline)
        return self.enc_len + self.dec_len


class Attention(nn.Module):
    """Multi-head attention. Self-attention when context is None (optionally with RoPE),
    cross-attention when context is given (queries from x, keys/values from context)."""

    def __init__(self, d_model, n_heads, dropout, rope_len=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.d_head, self.dropout = n_heads, d_model // n_heads, dropout
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.use_rope = rope_len is not None
        if self.use_rope:
            cos, sin = build_rope_cache(rope_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, context=None, causal=False):
        B, T, C = x.shape
        src = x if context is None else context
        S = src.size(1)
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)          # (B,H,T,Dh)
        k, v = self.kv_proj(src).view(B, S, 2, self.n_heads, self.d_head).permute(2, 0, 3, 1, 4)
        if self.use_rope:
            # cast the cache to q's dtype so q, k, v share one dtype under fp16 autocast
            q = apply_rope(q, self.rope_cos[:T].to(q.dtype), self.rope_sin[:T].to(q.dtype))
            k = apply_rope(k, self.rope_cos[:S].to(k.dtype), self.rope_sin[:S].to(k.dtype))
        out = F.scaled_dot_product_attention(q, k, v, is_causal=causal,
                                             dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(out.transpose(1, 2).reshape(B, T, C))


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model),
                                 nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = Attention(cfg.d_model, cfg.n_heads, cfg.dropout, rope_len=cfg.enc_len)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):                                   # pre-LN, bidirectional
        x = x + self.drop(self.attn(self.ln1(x)))
        return x + self.ffn(self.ln2(x))


class DecoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = Attention(cfg.d_model, cfg.n_heads, cfg.dropout, rope_len=cfg.dec_len)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.cross_attn = Attention(cfg.d_model, cfg.n_heads, cfg.dropout)          # no RoPE
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, y, memory):                           # pre-LN
        y = y + self.drop(self.self_attn(self.ln1(y), causal=True))
        y = y + self.drop(self.cross_attn(self.ln2(y), context=memory))
        return y + self.ffn(self.ln3(y))


class EncDecLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(cfg.vocab_size, cfg.d_model)       # shared by encoder, decoder, head
        self.emb_drop = nn.Dropout(cfg.dropout)
        self.encoder = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.enc_layers)])
        self.enc_norm = nn.LayerNorm(cfg.d_model)
        self.decoder = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.dec_layers)])
        self.dec_norm = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.emb.weight                          # weight tying

    def encode(self, src):
        """src (B, S<=enc_len) -> encoder hidden states (B, S, d). Also the Phase-2 feature extractor."""
        x = self.emb_drop(self.emb(src))
        for layer in self.encoder:
            x = layer(x)
        return self.enc_norm(x)

    def decode(self, memory, dec_in):
        """memory (B, S, d), dec_in (B, T<=dec_len) -> next-token logits (B, T, V)."""
        y = self.emb_drop(self.emb(dec_in))
        for layer in self.decoder:
            y = layer(y, memory)
        return self.head(self.dec_norm(y))

    def split(self, x):
        """Contiguous window x (B, enc_len+dec_len) -> encoder input, decoder input, decoder targets."""
        E = self.cfg.enc_len
        src, tgt = x[:, :E], x[:, E:]
        bos = torch.full((x.size(0), 1), self.cfg.bos_id, dtype=x.dtype, device=x.device)
        dec_in = torch.cat([bos, tgt[:, :-1]], dim=1)               # teacher forcing, shifted right
        return src, dec_in, tgt

    def forward(self, x):
        src, dec_in, _ = self.split(x)
        return self.decode(self.encode(src), dec_in)

    def compute_loss(self, x, y=None):
        """Cross-entropy on the continuation (decoder) tokens only. y is unused: targets come from x."""
        src, dec_in, tgt = self.split(x)
        logits = self.decode(self.encode(src), dec_in)
        return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), tgt.reshape(-1))

In [ ]:
%%writefile a3src/prefixlm.py
# Architecture 3: Prefix-LM (hybrid encoder/decoder in ONE stack) with RMSNorm, SwiGLU, RoPE, KV cache.
#
# One Transformer stack, two attention patterns decided by a prefix length p:
#   * prefix tokens   [0, p)  attend to EVERY prefix token (bidirectional, like an encoder)
#   * continuation    [p, T)  attend causally to everything before them (like a decoder)
#   mask: token i may attend to token j  <=>  (j <= i) or (j < p)
# Loss: only on predictions that cannot see their own target, i.e. positions i >= p-1.
# p is sampled per sequence during training (p = 0 -> plain causal LM), so the model learns both
# "read a prompt bidirectionally, then continue" and ordinary language modelling.
# Generation: the prompt is the prefix (one bidirectional "prefill" pass that fills a KV cache),
# then each new token is a single-position step that reuses the cached keys/values.
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


@dataclass
class PrefixLMConfig:
    vocab_size: int
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    n_kv_heads: int = 12          # < n_heads -> grouped-query attention (smaller KV cache)
    ffn_hidden: int = 2048        # SwiGLU hidden size (~8/3 * d_model keeps params like a 4x GELU FFN)
    max_seq_len: int = 1024
    dropout: float = 0.0
    prefix_min: float = 0.0       # training prefix length p ~ U[prefix_min*T, prefix_max*T]
    prefix_max: float = 0.85
    eval_prefix: float = 0.75     # fixed p for validation loss (deterministic, task-like)


class RMSNorm(nn.Module):
    """x / sqrt(mean(x^2) + eps) * g  - no mean subtraction, no bias (cheaper than LayerNorm)."""

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return xf.to(x.dtype) * self.weight


class SwiGLU(nn.Module):
    """FFN(x) = W2( SiLU(W1 x) * (W3 x) )  - gated feed-forward (PaLM / LLaMA)."""

    def __init__(self, d_model, hidden, dropout):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden, bias=False)
        self.w3 = nn.Linear(d_model, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d_model, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
        self.h, self.kvh, self.dh = cfg.n_heads, cfg.n_kv_heads, cfg.d_model // cfg.n_heads
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.dh, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kvh * self.dh, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kvh * self.dh, bias=False)
        self.o_proj = nn.Linear(self.h * self.dh, cfg.d_model, bias=False)
        self.dropout = cfg.dropout

    def forward(self, x, cos, sin, attn_mask=None, cache=None):
        """x (B,T,C); cos/sin for these T positions; attn_mask bool (B,1,T,S) True=attend or None=all;
        cache = (k, v) of earlier positions or None. Returns output and the updated (k, v)."""
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kvh, self.dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kvh, self.dh).transpose(1, 2)
        q = apply_rope(q, cos.to(q.dtype), sin.to(q.dtype))
        k = apply_rope(k, cos.to(k.dtype), sin.to(k.dtype))
        if cache is not None:
            k = torch.cat([cache[0], k], dim=2)
            v = torch.cat([cache[1], v], dim=2)
        new_cache = (k, v)
        if self.kvh != self.h:                                   # GQA: share each K/V head
            k = k.repeat_interleave(self.h // self.kvh, dim=1)
            v = v.repeat_interleave(self.h // self.kvh, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                             dropout_p=self.dropout if self.training else 0.0)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1)), new_cache


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = Attention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg.d_model, cfg.ffn_hidden, cfg.dropout)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin, attn_mask=None, cache=None):     # pre-norm residual block
        a, new_cache = self.attn(self.norm1(x), cos, sin, attn_mask, cache)
        x = x + self.drop(a)
        return x + self.ffn(self.norm2(x)), new_cache


class PrefixLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight                      # weight tying
        cos, sin = build_rope_cache(cfg.max_seq_len, cfg.d_model // cfg.n_heads)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    @staticmethod
    def prefix_mask(prefix_len, T):
        """prefix_len (B,) -> bool mask (B,1,T,T): i attends j  <=>  j <= i  or  j < p."""
        i = torch.arange(T, device=prefix_len.device)
        causal = i[None, :] <= i[:, None]                              # (T,T)  [i, j]: j <= i
        in_prefix = i[None, None, :] < prefix_len[:, None, None]       # (B,1,T): j < p
        return (causal[None] | in_prefix)[:, None]                     # (B,1,T,T)

    def _run(self, idx, attn_mask, start_pos=0, caches=None):
        T = idx.size(1)
        cos = self.rope_cos[start_pos:start_pos + T]
        sin = self.rope_sin[start_pos:start_pos + T]
        x = self.drop(self.tok_emb(idx))
        new_caches = []
        for li, block in enumerate(self.blocks):
            x, c = block(x, cos, sin, attn_mask, None if caches is None else caches[li])
            new_caches.append(c)
        return self.norm(x), new_caches

    def hidden(self, idx, prefix_len=None):
        """Final hidden states (B,T,d). Default = fully bidirectional (whole input is the prefix):
        the natural feature extractor for Phase 2+ (NER, classification, ...)."""
        B, T = idx.shape
        p = torch.full((B,), T if prefix_len is None else prefix_len, device=idx.device)
        return self._run(idx, self.prefix_mask(p, T))[0]

    def forward(self, idx, prefix_len=None):
        return self.head(self.hidden(idx, 0 if prefix_len is None else prefix_len))

    def compute_loss(self, x, y):
        """Next-token loss on positions i >= p-1 (positions that cannot see their target)."""
        B, T = x.shape
        if self.training:
            lo, hi = int(self.cfg.prefix_min * T), int(self.cfg.prefix_max * T)
            p = torch.randint(lo, hi + 1, (B,), device=x.device)
        else:
            p = torch.full((B,), int(self.cfg.eval_prefix * T), device=x.device)
        h, _ = self._run(x, self.prefix_mask(p, T))
        logits = self.head(h)
        pos = torch.arange(T, device=x.device)[None, :]
        target = y.masked_fill(pos < (p[:, None] - 1).clamp(min=0), -100)   # ignore leaky positions
        return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), target.reshape(-1),
                               ignore_index=-100)

    # ---------------- generation with a KV cache ----------------
    @torch.no_grad()
    def prefill(self, prompt):
        """Prompt (B,P) is the prefix: one bidirectional pass. Returns next-token logits (B,V) + cache."""
        h, caches = self._run(prompt, None)                       # None mask = every prompt token sees all
        return self.head(h[:, -1]), caches

    @torch.no_grad()
    def decode_step(self, token, caches, pos):
        """token (B,1) at absolute position pos; attends to all cached positions + itself."""
        h, caches = self._run(token, None, start_pos=pos, caches=caches)
        return self.head(h[:, -1]), caches

In [ ]:
%%writefile a3src/a1_tokenizer.py
"""Assignment-1 byte-level BPE tokenizer, retrained on the Assignment-3 corpus.

Byte-level BPE (A-1 Q5 spec): byte symbols so no <unk> is needed, reserved tokens
<pad> <unk> <s> </s> at ids 0..3, each merge takes the most frequent adjacent pair
(ties broken by the lexicographically smallest merged bytes), and merges never cross
a pre-token (word / whitespace) boundary so decode() reconstructs the text exactly.
"""
import re
from collections import Counter

RESERVED = ("<pad>", "<unk>", "<s>", "</s>")
BASE = len(RESERVED)
SPLIT = re.compile(r"\S+|\s+")


def merge_sequence(ids, pair, new_id):
    a, b = pair
    out = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == a and ids[i + 1] == b:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return tuple(out)


class BPETokenizer:
    def __init__(self):
        self.merges = {}
        self.ranks = {}
        self.itos = {}
        self.vocab_size = 0
        self.cache = {}

    def train(self, text, vocab_size):
        self.itos = {i: b"" for i in range(BASE)}
        for byte in range(256):
            self.itos[BASE + byte] = bytes((byte,))

        counts = Counter(SPLIT.findall(text))
        corpus = Counter()
        for word, freq in counts.items():
            ids = tuple(BASE + byte for byte in word.encode("utf-8"))
            corpus[ids] += freq

        n_merges = max(0, vocab_size - BASE - 256)
        next_id = BASE + 256
        while len(self.merges) < n_merges:
            pair_freq = Counter()
            for ids, freq in corpus.items():
                for j in range(len(ids) - 1):
                    pair_freq[(ids[j], ids[j + 1])] += freq
            if not pair_freq:
                break
            top = max(pair_freq.values())
            if top < 2:
                break
            best_pairs = [p for p, f in pair_freq.items() if f == top]
            pair = min(best_pairs, key=lambda p: self.itos[p[0]] + self.itos[p[1]])

            self.itos[next_id] = self.itos[pair[0]] + self.itos[pair[1]]
            self.merges[pair] = next_id
            self.ranks[pair] = len(self.ranks)

            new_corpus = Counter()
            for ids, freq in corpus.items():
                new_corpus[merge_sequence(ids, pair, next_id)] += freq
            corpus = new_corpus
            next_id += 1

        self.vocab_size = BASE + 256 + len(self.merges)

    def encode_chunk(self, raw_bytes):
        if raw_bytes in self.cache:
            return self.cache[raw_bytes]
        ids = [BASE + byte for byte in raw_bytes]
        while len(ids) >= 2:
            candidates = {}
            for j in range(len(ids) - 1):
                pair = (ids[j], ids[j + 1])
                if pair in self.ranks:
                    candidates[pair] = self.ranks[pair]
            if not candidates:
                break
            pair = min(candidates, key=candidates.get)
            ids = list(merge_sequence(ids, pair, self.merges[pair]))
        self.cache[raw_bytes] = ids
        return ids

    def encode(self, text):
        ids = []
        for chunk in SPLIT.findall(text):
            ids.extend(self.encode_chunk(chunk.encode("utf-8")))
        return ids

    def decode(self, ids):
        data = b""
        for i in ids:
            data += self.itos.get(i, b"")
        return data.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self.vocab_size


def train_tokenizer(train_text, vocab_size):
    tokenizer = BPETokenizer()
    tokenizer.train(train_text, vocab_size)
    return tokenizer

In [ ]:
%%writefile a3src/space_bpe.py
# Space-attached byte-level BPE (GPT-2 style pre-tokenisation) built on your A1 tokenizer.
#
# Difference from A1: A1 splits text into  \S+ | \s+  so every space is its own token
# (~2.3 tokens per word). Here a single leading space is attached to the following word
# (" the", " Court"), so a word usually costs ~1.3 tokens and a 512-word prompt fits in the
# 1024-token context. Merge rule is unchanged (most frequent pair, ties -> lexicographically
# smallest merged bytes, never across a pre-token boundary), so decode() is still exact.
# train() is an incremental version of the A1 algorithm: it only re-counts pairs in words that
# contain the merged pair (plus a heap), which gives the same merges but runs minutes, not hours.
import heapq
import re
from collections import Counter, defaultdict
from a1_tokenizer import BPETokenizer, BASE, merge_sequence

# " ?\S+"      a word with at most one leading space
# "\s+(?!\S)"  a whitespace run, leaving its last space for the next word
# "\s+"        trailing whitespace at the very end
SPACE_SPLIT = re.compile(r" ?\S+|\s+(?!\S)|\s+")


class SpaceBPETokenizer(BPETokenizer):
    space_prefix = True          # tells the pipeline not to append a trailing space to prompts

    def train(self, text, vocab_size):
        self.itos = {i: b"" for i in range(BASE)}
        for byte in range(256):
            self.itos[BASE + byte] = bytes((byte,))
        counts = Counter(SPACE_SPLIT.findall(text))
        words = [tuple(BASE + b for b in w.encode("utf-8")) for w in counts]
        freqs = list(counts.values())

        pair_counts = defaultdict(int)          # pair -> total frequency
        where = defaultdict(set)                # pair -> indices of words containing it
        for wi, ids in enumerate(words):
            for pair in zip(ids, ids[1:]):
                pair_counts[pair] += freqs[wi]
                where[pair].add(wi)
        merged = lambda p: self.itos[p[0]] + self.itos[p[1]]
        heap = [(-c, merged(p), p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        n_merges = max(0, vocab_size - BASE - 256)
        next_id = BASE + 256
        while len(self.merges) < n_merges and heap:
            neg_c, _, pair = heapq.heappop(heap)
            c = pair_counts.get(pair, 0)
            if c != -neg_c or c <= 0:
                continue                         # stale heap entry
            if c < 2:
                break
            self.itos[next_id] = merged(pair)
            self.merges[pair] = next_id
            self.ranks[pair] = len(self.ranks)
            delta = defaultdict(int)
            for wi in where.pop(pair, ()):
                old = words[wi]
                new = merge_sequence(old, pair, next_id)
                if new == old:
                    continue
                f = freqs[wi]
                for p in zip(old, old[1:]):
                    delta[p] -= f
                for p in zip(new, new[1:]):
                    delta[p] += f
                    where[p].add(wi)
                words[wi] = new
            pair_counts.pop(pair, None)
            for p, d in delta.items():
                if d == 0 or p == pair:
                    continue
                pair_counts[p] += d
                if pair_counts[p] > 0:
                    heapq.heappush(heap, (-pair_counts[p], merged(p), p))
            next_id += 1
        self.vocab_size = BASE + 256 + len(self.merges)

    def encode(self, text):
        ids = []
        for chunk in SPACE_SPLIT.findall(text):
            ids.extend(self.encode_chunk(chunk.encode("utf-8")))
        return ids


def train_space_tokenizer(train_text, vocab_size):
    tok = SpaceBPETokenizer()
    tok.train(train_text, vocab_size)
    return tok

In [ ]:
# 4b · Imports, tokenizer save/load, AMP helper ---------------------------------
import sys, importlib
sys.path.insert(0, os.path.abspath("a3src"))
import rope, attention, model_arch, a1_tokenizer, encdec, prefixlm, space_bpe
for _m in (rope, attention, model_arch, a1_tokenizer, encdec, prefixlm, space_bpe): importlib.reload(_m)
from a1_tokenizer import BPETokenizer, train_tokenizer
from model_arch import LMConfig, DecoderLM
from encdec import EncDecConfig, EncDecLM
from prefixlm import PrefixLMConfig, PrefixLM
from space_bpe import SpaceBPETokenizer, train_space_tokenizer

def save_tokenizer(tok, path):
    cache = tok.cache; tok.cache = {}            # don't pickle the (large) encode cache
    try:
        with open(path, "wb") as f: pickle.dump(tok, f)
    finally:
        tok.cache = cache

def load_tokenizer(path):
    with open(path, "rb") as f: return pickle.load(f)

PAD, UNK, BOS, EOS = 0, 1, 2, 3                  # reserved ids in your A1 tokenizer
_use_cuda = (device == "cuda")
def amp_autocast():
    return torch.autocast(device_type=("cuda" if _use_cuda else "cpu"), enabled=_use_cuda)

DATA_READY = os.path.exists(os.path.join(DATA_DIR, "meta.json"))
print("Imported your A3 DecoderLM + A1 BPE tokenizer. Prebuilt data found:", DATA_READY)

## 5 · Data: stream → document-level split → dev set
Skipped automatically if `DATA_DIR` already has `meta.json` (for example when `prebuilt_dir` points to a saved Kaggle dataset).
**Look at the printed samples**: are they real legal text? Do the dev prompts look like the official test prompts?

In [ ]:
# 5a · Discover files, download only the (small) test file ---------------------
from huggingface_hub import HfApi, HfFileSystem, hf_hub_download
api = HfApi()
repo_files  = api.list_repo_files(CONFIG["hf_repo"], repo_type="dataset")
phase_files = [f for f in repo_files if f.startswith(CONFIG["phase_dir"] + "/")]
print("Phase-1 files:", *phase_files, sep="\n  ")

train_files = {}
for src in CONFIG["sources"]:
    m = [f for f in phase_files if "/Train/" in f and src in f.lower() and f.endswith(".json")]
    if m: train_files[src] = m[0]
test_files = [f for f in phase_files if "/Test/" in f and f.endswith(".json")]
assert train_files and test_files, "Phase-1 train/test files not found."
test_path = hf_hub_download(CONFIG["hf_repo"], test_files[0], repo_type="dataset")
print("Train sources:", list(train_files), "| test file:", test_files[0])

In [ ]:
# 5b · Stream documents (no full download) and build the splits ------------------
import ijson
print("ijson backend:", ijson.backend)           # yajl2_c (C) is much faster than 'python'
fs = HfFileSystem()

def hf_open(fname):
    return fs.open(f"datasets/{CONFIG['hf_repo']}/{fname}", "rb", block_size=16 * 1024 * 1024)

def iter_texts(obj):
    """Yield strings from a record of unknown shape (dict/list/str)."""
    if isinstance(obj, str):
        if obj.strip(): yield obj
    elif isinstance(obj, dict):
        for k in ("text", "content", "body", "document", "judgment", "raw_text"):
            if isinstance(obj.get(k), str) and obj[k].strip():
                yield obj[k]; return
        for v in obj.values():
            yield from iter_texts(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from iter_texts(v)

def toplevel_kind(fname):
    with hf_open(fname) as f:
        while True:
            c = f.read(1)
            if not c: return None
            if c.strip(): return c                # b"{" or b"["

def stream_docs(fname, max_chars):
    kind, got = toplevel_kind(fname), 0
    with hf_open(fname) as f:
        gen = ijson.kvitems(f, "") if kind == b"{" else ijson.items(f, "item")
        for item in gen:
            val = item[1] if kind == b"{" else item
            for t in iter_texts(val):
                yield t; got += len(t)
                if got >= max_chars: return

def collect_docs(fname, budget, tries=3):
    """Streaming ~200 MB over the network can hiccup; retry the whole source (same docs every time)."""
    for attempt in range(1, tries + 1):
        try:
            return list(stream_docs(fname, budget))
        except Exception as e:
            print(f"  stream error on {fname} ({type(e).__name__}: {e}); retry {attempt}/{tries}")
            time.sleep(15)
    raise RuntimeError(f"could not stream {fname}")

def build_dev_set(docs, n, prompt_words=512, cont_words=128, seed=0):
    """Mimic the official task: a 512-word prompt and the next 128 words as reference."""
    rng = random.Random(seed)
    need = prompt_words + cont_words
    pools = [d.split() for d in docs]
    pools = [w for w in pools if len(w) >= need] or [sum(pools, [])]   # fallback: concatenate
    prompts, refs = {}, {}
    for i in range(n):
        w = rng.choice(pools)
        s = rng.randint(0, len(w) - need)
        prompts[str(i)] = {"text": " ".join(w[s:s + prompt_words])}
        refs[str(i)]    = {"text": " ".join(w[s + prompt_words:s + need])}
    return prompts, refs

if not DATA_READY:
    CHARS_PER_TOKEN_GUESS = 4.8 if CONFIG["tokenizer_type"] == "space" else 6   # generous; true value printed later
    char_budget = max(TRAIN_TOKENS * CHARS_PER_TOKEN_GUESS, CONFIG["tok_train_chars"] * 1.2) / (1 - CONFIG["val_doc_frac"])
    wsum = sum(CONFIG["mix"][s] for s in train_files)
    rng = random.Random(CONFIG["seed"])
    train_pairs, val_by_src, source_stats = [], {}, {}
    for src, fname in train_files.items():
        budget = int(char_budget * CONFIG["mix"][src] / wsum)
        t0 = time.time()
        docs = collect_docs(fname, budget)
        rng.shuffle(docs)
        n_val = max(1, int(round(len(docs) * CONFIG["val_doc_frac"])))
        val_by_src[src] = docs[:n_val]                      # held out PER SOURCE (stratified)
        train_pairs += [(src, d) for d in docs[n_val:]]
        source_stats[src] = dict(docs=len(docs), chars=sum(map(len, docs)), val_docs=n_val)
        print(f"{src:>14}: {len(docs):>7,} docs | {source_stats[src]['chars']:>13,} chars | "
              f"{n_val} held out | {time.time()-t0:.0f}s")
    rng.shuffle(train_pairs)
    train_src = [s for s, _ in train_pairs]; train_docs = [d for _, d in train_pairs]; del train_pairs
    print(f"\nTrain docs: {len(train_docs):,} | Val docs: {sum(map(len, val_by_src.values())):,}")
    print("\n--- SAMPLE train doc ---\n", train_docs[0][:500])
    os.makedirs(DATA_DIR, exist_ok=True)
    # dev set: equal number of examples from every source's held-out docs
    dev_prompts, dev_refs, per = {}, {}, max(1, CONFIG["n_dev_examples"] // len(val_by_src))
    for j, (src, vdocs) in enumerate(val_by_src.items()):
        p, r = build_dev_set(vdocs, per, seed=CONFIG["seed"] + j)
        for k in p:
            key = str(len(dev_prompts))
            dev_prompts[key] = {**p[k], "source": src}; dev_refs[key] = {**r[k], "source": src}
    json.dump(dev_prompts, open(os.path.join(DATA_DIR, "dev_prompts.json"), "w"), ensure_ascii=False)
    json.dump(dev_refs,    open(os.path.join(DATA_DIR, "dev_refs.json"), "w"), ensure_ascii=False)
    print("\n--- SAMPLE dev prompt (last 200 chars) ---\n", dev_prompts["0"]["text"][-200:])
    print("--- its reference (first 200 chars) ---\n", dev_refs["0"]["text"][:200])

In [ ]:
# 6 · Train (or load) your byte-level BPE tokenizer -----------------------------
tok_path = os.path.join(DATA_DIR, "legal_bpe.pkl")
if os.path.exists(tok_path):
    tokenizer = load_tokenizer(tok_path)
    print("Loaded tokenizer from", tok_path)
else:
    parts, n = [], 0                               # train-split docs only (no val leakage)
    for d in train_docs:
        parts.append(d); n += len(d) + 2
        if n >= CONFIG["tok_train_chars"]: break
    tok_text = "\n\n".join(parts)[:CONFIG["tok_train_chars"]]
    print(f"Training {CONFIG['tokenizer_type']} BPE (vocab={CONFIG['vocab_size']}) on {len(tok_text):,} chars ...")
    t0 = time.time()
    trainer = train_space_tokenizer if CONFIG["tokenizer_type"] == "space" else train_tokenizer
    tokenizer = trainer(tok_text, CONFIG["vocab_size"])
    save_tokenizer(tokenizer, tok_path)
    print(f"Done in {time.time()-t0:.0f}s -> {tok_path}")

VOCAB = tokenizer.get_vocab_size()
import hashlib                                    # fingerprint: detects a mismatched tokenizer/checkpoint
TOK_HASH = hashlib.md5(json.dumps(sorted([a, b, c] for (a, b), c in tokenizer.merges.items())).encode()).hexdigest()[:12]
print("Tokenizer fingerprint:", TOK_HASH)
assert VOCAB < 65535, "token ids are stored as uint16"
s = "In the Supreme Court of India, the appellant contended that Section 302 IPC"
print("Vocab size:", VOCAB, "| round-trip OK:", tokenizer.decode(tokenizer.encode(s)) == s)
print("Tokenizer:", type(tokenizer).__name__, "| pieces:", [tokenizer.decode([i]) for i in tokenizer.encode(s)])
print(f"Tokens per word on this sentence: {len(tokenizer.encode(s)) / len(s.split()):.2f}")

In [ ]:
# 6b · Encode the corpus in parallel -> train.bin / val.bin (uint16) -----------
from multiprocessing import Pool

def _encode_doc(doc):
    ids = tokenizer.encode(doc); ids.append(EOS)
    return np.asarray(ids, dtype=np.uint16)

def encode_docs(docs, max_tokens, workers=os.cpu_count() or 2):
    """Returns (token array, chars consumed, per-doc token counts of the docs used)."""
    out, n, chars, lens, t0 = np.empty(max_tokens, dtype=np.uint16), 0, 0, [], time.time()
    with Pool(workers) as pool:
        for i, arr in enumerate(pool.imap(_encode_doc, docs, chunksize=8)):
            k = min(len(arr), max_tokens - n)
            out[n:n + k] = arr[:k]; n += k; chars += len(docs[i]); lens.append(k)
            if i % 2000 == 0:
                print(f"  {i:,} docs -> {n:,} tokens ({time.time()-t0:.0f}s)")
            if n >= max_tokens: break
    return out[:n], chars, lens

if not DATA_READY:
    # validation: equal token budget per source, stored as contiguous blocks (ranges saved in meta)
    val_parts, val_ranges, pos = [], {}, 0
    for src, vdocs in val_by_src.items():
        arr, _, _ = encode_docs(vdocs, CONFIG["val_tokens"] // len(val_by_src))
        val_parts.append(arr); val_ranges[src] = [pos, pos + len(arr)]; pos += len(arr)
    val_arr = np.concatenate(val_parts)
    train_arr, train_chars, lens = encode_docs(train_docs, TRAIN_TOKENS)
    train_tokens_by_source = {s: 0 for s in val_by_src}
    for s, k in zip(train_src, lens): train_tokens_by_source[s] += k
    train_arr.tofile(os.path.join(DATA_DIR, "train.bin"))
    val_arr.tofile(os.path.join(DATA_DIR, "val.bin"))
    meta = dict(vocab_size=VOCAB, n_train=int(len(train_arr)), n_val=int(len(val_arr)),
                chars_per_token=round(train_chars / max(1, len(train_arr)), 3),
                train_tokens_by_source=train_tokens_by_source, val_ranges=val_ranges,
                mix=CONFIG["mix"], source_stats=source_stats, val_doc_frac=CONFIG["val_doc_frac"],
                tok_train_chars=CONFIG["tok_train_chars"], seed=CONFIG["seed"], tok_hash=TOK_HASH,
                train_tokens_budget=TRAIN_TOKENS)
    json.dump(meta, open(os.path.join(DATA_DIR, "meta.json"), "w"), indent=2)
    del train_docs, train_src, train_arr, val_arr, val_parts, val_by_src   # free RAM
    DATA_READY = True

meta = json.load(open(os.path.join(DATA_DIR, "meta.json")))
train_data = np.memmap(os.path.join(DATA_DIR, "train.bin"), dtype=np.uint16, mode="r")
val_data   = np.memmap(os.path.join(DATA_DIR, "val.bin"),   dtype=np.uint16, mode="r")
dev_prompts = json.load(open(os.path.join(DATA_DIR, "dev_prompts.json"), encoding="utf-8"))
dev_refs    = json.load(open(os.path.join(DATA_DIR, "dev_refs.json"),    encoding="utf-8"))
epochs = CONFIG["max_steps"] * TOKENS_PER_STEP / len(train_data)
print(f"Train tokens: {len(train_data):,} | Val tokens: {len(val_data):,} | "
      f"chars/token: {meta['chars_per_token']} | dev examples: {len(dev_prompts)}")
print(f"This run will make {epochs:.2f} passes over the training tokens.")
if epochs > 1.05: print("NOTE: >1 pass means repeated data (this architecture uses more steps/tokens than arch-1).")
assert meta.get("tok_hash") in (None, TOK_HASH), "tokenizer does not match the one this data was encoded with"
for src, n in meta.get("train_tokens_by_source", {}).items():
    r = meta.get("val_ranges", {}).get(src, [0, 0])
    print(f"  {src:>14}: train {n:>12,} tokens ({n / max(1, len(train_data)):.1%}) | val {r[1]-r[0]:>10,} tokens")